In [ ]:
import pandas as pd
import pickle

from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from scikit_posthocs import posthoc_nemenyi_friedman, critical_difference_diagram

import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme()

In [ ]:
# Load real dataset results
with open('data_real.pkl', "rb") as f:
    real = pickle.load(f)['kwargs']['df']

# Load synthetic dataset results
with open('data_synthetic.pkl', "rb") as f:
    synth = pickle.load(f)['kwargs']['df']

results_df = pd.concat([real, synth], axis=1)

In [ ]:
# Calculate the number dataframe index runs level
num_runs = results_df.index.get_level_values('run').nunique()

# Reorder MultiIndex columns for consistent plotting
order = {
    'dataset': [
        '197_cpu_act', '4544_GOM', '505_tecator', 'CCN', 'CCUN',
        'F1', 'F2', 'Friedman1', 'Friedman2', 'Friedman3'
    ],
    'method':  ['GPPI', 'GPSHAP', 'GPCMI', 'RFGPCMI', 'GP'],
    'metric':  ['training_losses', 'validation_losses', 'test_losses']
}

# Define colors for each method
colors = sns.color_palette(n_colors=len(order['method']))
hue = dict(zip(order['method'], colors))

for level, cols in order.items():
    results_df = results_df.reindex(columns=cols, level=level)

results_df.tail(3)

In [ ]:
# Calculate the median test loss for each method across all datasets
median_test_loss_df = results_df.xs('test_losses', level='metric', axis=1).groupby('run').last().median().unstack(level='method')
median_test_loss_df

In [ ]:
print(" - Mean Ranks - ")
avg_ranks = median_test_loss_df.rank(axis=1).mean()
print(avg_ranks)

stat, p_value = friedmanchisquare(
    median_test_loss_df['GP'], 
    median_test_loss_df['GPPI'], 
    median_test_loss_df['GPSHAP'], 
    median_test_loss_df['GPCMI'], 
    median_test_loss_df['RFGPCMI']
)

print("\n - Friedman Test - ")
print(f"Friedman statistic: {stat:.5f}")
print(f"p-value: {p_value:.5f}")


In [ ]:
methods = [m for m in order['method'] if m != 'RFGPCMI']
results = {}
alpha = 0.05

print("\n - Wilcoxon Signed-Rank Test - ")
for m in methods:
    stat, p = wilcoxon(
        x = median_test_loss_df['RFGPCMI'], 
        y = median_test_loss_df[m],
        method = 'exact'
    )
    results[m] = (stat, p)
    print(f"{m:<6}: W={stat:.3f}, p-value={p:.5f}, rejected={p < alpha}")

print("\n - Holm-Bonferroni Correction - ")
pvals = [results[m][1] for m in methods]
reject, pvals_corrected, _, _ = multipletests(pvals, alpha=alpha, method='holm')

for m, p_corr, rej in zip(methods, pvals_corrected, reject):
    print(f"{m:<6}: p-value={p_corr:.5f}, rejected={rej}")

In [ ]:
nemenyi_pvalues = posthoc_nemenyi_friedman(median_test_loss_df.values)
nemenyi_pvalues.columns = order['method']
nemenyi_pvalues.index = order['method']

plt.figure(figsize=(8, 3))
critical_difference_diagram(avg_ranks, nemenyi_pvalues, hue=hue, alpha=alpha)
plt.title("Critical Difference Diagram (Friedman + Nemenyi)")
plt.show()